**模型**（model）是智能体的大脑，负责推理分析。

**工具**（tools）是智能体的手脚，负责执行任务，与外界交互

定义一个带有工具的Agent的基本流程如下：

定义工具

初始化模型

初始化Agent，绑定模型和工具

1.自定义工具

所谓的工具（**tools**），本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。

包括以下信息：

工具名

工具的作用

工具需要的参数

1.1.基于tool描述工具

可以通过**装饰器**来定义工具名、工具的作用

In [1]:
from langchain_core.tools import tool

@tool("square_root", description = "Calculate the square root of a number")
def tool(x: float) -> float:
    return x ** 0.5

1.2.使用**函数名**和**文档注释**描述工具

如果不@tool装饰器没有定义工具名和描述作用，此时：

工具名：默认就是函数名

工具所需的参数： 默认就是函数的参数列表

工具作用的描述： 默认就是函数的文档注释

In [8]:
from langchain_core.tools import tool

@tool
def square_root(x: float) -> float:
    """
    Calculate the square root of a number
    """
    return x ** 0.5

In [9]:
response = square_root.invoke({"x": 255})
print(response)

15.968719422671311


In [4]:
# 定义一个查询天气的tool

@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    get current weather and optional forecast.
    args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

1.3定义Pydantic Model 描述参数

In [6]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal

# 查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数"""
    location: str = Field(description = "City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default = "celsius",
        description = "Temperature unit preference"
    )
    include_forecast: bool = Field(
        default = False,
        description = "Include 5-day forecast"
    )


@tool(args_schema = WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    get current weather and optional forecast.
    """
    temp = 22 if units == "celsius" else 72
    result = f"current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [11]:
response = get_weather.invoke({"location": "杭州", "include_forecast": True})
print(response)

current weather in 杭州: 22 degrees C
Next 5 days: Sunny


测试

In [13]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model = "deepseek-v4-pro",
    tools = [square_root, get_weather]
)

In [14]:
for token, metadata in agent.stream(
        {"messages": [HumanMessage(content = "杭州接下来的几天天气如何？")]},
        stream_mode = "messages"
):
    print(token.content, end = "", flush = True)

好的，我来帮你查询杭州未来几天的天气。current weather in 杭州: 22 degrees C
Next 5 days: Sunny杭州的天气情况如下：

- **当前气温**：22°C
- **未来5天天气**：晴天 ☀️

总体来看，杭州接下来几天天气不错，以晴好为主，非常适合出行和户外活动。不过早晚温差可能会稍大，建议出门时备一件薄外套。

In [17]:
response = agent.invoke(
    {"messages": [HumanMessage(content = "467和255的平方根是多少 ？")]}
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和255的平方根是多少 ？
None
================================== Ai Message ==================================

我来计算467和255的平方根。
Tool Calls:
  square_root (call_00_qVfVW7QMln12gX7JCv8n1931)
 Call ID: call_00_qVfVW7QMln12gX7JCv8n1931
  Args:
    x: 467
  square_root (call_01_pAEvdX0LOcZJY7giH7cc9404)
 Call ID: call_01_pAEvdX0LOcZJY7giH7cc9404
  Args:
    x: 255
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

15.968719422671311
None
================================== Ai Message ==================================

计算结果如下：

- **467 的平方根** ≈ **21.6102**
- **255 的平方根** ≈ **15.9687**

如果需要更高精度或进行其他运算，请告诉我！
None


2.预定义Tool

LangChain中提供了很多的预定义的Tool，tavily就是一个用来做Web搜索的工具

2.1基本用法

创建账号，创建API_KEY

配置环境变量：TAVILY_API_KEY

安装依赖： uv add langchain_tavily

In [28]:
from langchain_tavily import TavilySearch
from dotenv import load_dotenv

load_dotenv()

search_tool = TavilySearch(
    max_results = 5,
    topic = "general",
)

In [22]:
search_tool.invoke("鸡你太美是什么梗")

{'query': '鸡你太美是什么梗',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://baike.baidu.com/item/%E9%B8%A1%E4%BD%A0%E5%A4%AA%E7%BE%8E/53592938',
   'title': '鸡你太美（网络流行语）_百度百科',
   'content': '01:23\n\n人民网评“鸡你太美”是恶俗烂梗:不能让恶俗的网络烂梗毒害孩子\n\n00:18\n\n“鸡你太美”遭恶搞,蔡徐坤曾发律师函\n\n00:21\n\n鸡你太美官方MV(全弹幕)\n\n02:54\n\n理智玩梗,抵制低俗烂梗,维护美好网络环境 "鸡你太美 "网评鸡你太美是恶俗烂梗 "你怎么看\n\n01:03\n\n订阅更新\n\n订阅更新\n\n300\n有用+1\n\n62\n\n鸡你太美，\n\n网络流行语\n，源于中国内地男子组合NPC成员蔡徐坤的一段选秀节目表演。为2016年11月29日\n\nSWIN-S\n发布的歌曲《\n\n只因你太美\n》的空耳。\n\n出自于\n\n蔡徐坤\n在综艺节目《\n\n偶像练习生\n》之中的自我介绍表演。\n\n \n原歌词为只因你太美，网友把只因听成了鸡。部分网友将其视频素材剪辑成鬼畜恶搞视频，鸡你太美随之逐渐成为网络热梗。\n\n \n\n鸡你太美在一定程度上是当代网友逆反心理的真实写照，其次，这个梗能够风靡不衰是其背后所携带的巨大流量。鸡你太美不断被创新，不断衍生出新的形式，吸引更多的网友前来围观。\n\n \n\n2020年1月，鸡你太美一词入选bilibili2019年度流行梗大赏。\n\n2023年3月7日， [...] 2023年3月7日，\n\n人民网\n评\n“鸡你太美”是恶俗烂梗。\n\n \n\n## 相关星图\n\n查看更多\n\n因发音问题形成的网络流行语\n\n共162个词条\n3.0万阅读\n\n空耳\n\n空耳，来源于日语词语“そらみみ”，原对应日语中“幻听”一词。指根据所听到原歌曲或原台词的发音，造出与之发音相似的另一句话，是一种对声音的再诠释。空耳是一种文字游戏，常以达到诙谐、恶搞或娱乐效果，并产生了如“阿姨洗铁路”、

In [24]:
# 创建智能体
agent = create_agent(
    model = "deepseek-v4-pro",
    tools = [search_tool],
    system_prompt = "你是一个智能助手，你使用工具来解决问题"
)

response = agent.invoke(
    {"messages": [HumanMessage(content = "鸡你太美是什么梗？")]}
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

鸡你太美是什么梗？
================================== Ai Message ==================================

"鸡你太美"是一个网络流行梗，让我帮你查一下详细的由来。
Tool Calls:
  tavily_search (call_00_9FwrgqTzlEilknmDBjwO5174)
 Call ID: call_00_9FwrgqTzlEilknmDBjwO5174
  Args:
    query: 鸡你太美 是什么梗
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "鸡你太美 是什么梗", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://baike.baidu.com/item/%E9%B8%A1%E4%BD%A0%E5%A4%AA%E7%BE%8E/53592938", "title": "鸡你太美（网络流行语）_百度百科", "content": "01:23\n\n人民网评“鸡你太美”是恶俗烂梗:不能让恶俗的网络烂梗毒害孩子\n\n00:18\n\n“鸡你太美”遭恶搞,蔡徐坤曾发律师函\n\n00:21\n\n鸡你太美官方MV(全弹幕)\n\n02:54\n\n理智玩梗,抵制低俗烂梗,维护美好网络环境 \"鸡你太美 \"网评鸡你太美是恶俗烂梗 \"你怎么看\n\n01:03\n\n订阅更新\n\n订阅更新\n\n300\n有用+1\n\n62\n\n鸡你太美，\n\n网络流行语\n，源于中国内地男子组合NPC成员蔡徐坤的一段选秀节目表演。为2016年11月29日\n\nSWIN-S\n发布的歌曲《\n\n只因你太美\n》的空耳。\n\n出自于\n\n蔡徐坤\n在综艺

2.2.优化

自定义tavily工具

In [40]:
# 先使用官方的客户端做初始化
tavily = TavilySearch(
    max_search = 5,
    topic = "general"
)


# 然后自己封装为tool
@tool
def web_search(query: str):
    """Search the web of information"""
    return tavily.invoke(query)

定义结构化输出实体

In [41]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description = "The title of the web page cited in the answer")
    url: str = Field(description = "The url of the web pag cited in the answer")

# Agent的回答内容
class AnswerInfo(BaseModel):
    answer: str = Field(description = "the final answer for user")
    reference: list[Reference] = Field(description = "the web pages cited in the answer")


In [46]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model = "deepseek-chat",
    tools = [web_search],
    system_prompt = "你是一个智能助手，你使用工具解决用户的问题",
    response_format = AnswerInfo
)


In [48]:
# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content = "鸡你太美是什么梗？")]},
)

# 获取结构化输出
print(response)

{'messages': [HumanMessage(content='鸡你太美是什么梗？', additional_kwargs={}, response_metadata={}, id='4a9d3c96-2afb-4c95-85b4-a2070325d194'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 441, 'total_tokens': 482, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 57}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': 'd2a25992-aa9f-4e5f-b068-0c04781e60c8', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe5cc-f863-7651-8c2f-b17c59a7c71b-0', tool_calls=[{'name': 'web_search', 'args': {'query': '鸡你太美是什么梗'}, 'id': 'call_00_mJIWWI5NKFlheNjIF28T0699', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 441, 'output_tokens': 41, 'total_t